# Small Object Detection and Tracking in UAV Imagery
YOLO fine-tuning on VisDrone, size-aware comparison of standard vs. sliced (SAHI) inference with COCO AP and bootstrap confidence intervals, ByteTrack tracking scored with py-motmetrics.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
!nvidia-smi

In [ ]:
GITHUB_USER = "mahdildarr"
!git clone https://github.com/{GITHUB_USER}/uav-small-object-detection.git
%cd uav-small-object-detection
!pip install -q -e .
!pytest -q

## 1. Training
VisDrone (~2 GB) is downloaded automatically on the first run. With `yolo11s` at 1024 px each epoch takes a few minutes on a T4. Override any setting with `--set`, e.g. `--set epochs=20 imgsz=640 name=quick`.

In [ ]:
!python -m uavdet.train --config configs/visdrone_yolo11s_1024.yaml

### Optional: slicing-aided fine-tuning
Train on 640 px tiles so that the training object scale matches sliced inference.

In [ ]:
# !python -m uavdet.slice_dataset --data VisDrone.yaml --out datasets/VisDrone-sliced --slice 640 --overlap 0.2
# !python -m uavdet.train --config configs/visdrone_sliced_finetune.yaml

## 2. Standard vs. sliced inference by object size
`--annotations` points to the raw VisDrone files so that ignored regions are honoured. The raw validation annotations are downloaded below.

In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip && unzip -q -o VisDrone2019-DET-val.zip -d raw
WEIGHTS = "runs/visdrone/yolo11s_1024/weights/best.pt"
!python -m uavdet.evaluate --weights {WEIGHTS} --images raw/VisDrone2019-DET-val/images \
    --annotations raw/VisDrone2019-DET-val/annotations --limit 0 --ap --save-vis 6 --out runs/eval/finetuned

In [ ]:
from IPython.display import Image, Markdown, display
import glob
display(Markdown(open("runs/eval/finetuned/report.md").read()))
display(Image("runs/eval/finetuned/recall_by_size.png", width=500))
for f in sorted(glob.glob("runs/eval/finetuned/vis/*.jpg"))[:3]:
    display(Image(f, width=1200))

## 3. Tracking on VisDrone-MOT
Download the VisDrone2019-MOT-val sequences from the [official repository](https://github.com/VisDrone/VisDrone-Dataset) (Google Drive / Baidu links) and set the path below.

In [ ]:
MOT = "/content/VisDrone2019-MOT-val"
for tracker in ["bytetrack.yaml", "botsort.yaml"]:
    name = tracker.split(".")[0]
    !python -m uavdet.track --weights {WEIGHTS} --sequences {MOT}/sequences --tracker {tracker} --out runs/track/{name}
    !python -m uavdet.eval_mot --results runs/track/{name} --annotations {MOT}/annotations

## Save results
Copy the outputs to Google Drive and update the results tables in the README.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/uav-small-object-detection
!cp -r runs /content/drive/MyDrive/uav-small-object-detection/